# Etap 06.05 — Model CNN (PyTorch)

Klasyfikacja wieloklasowa cyfr MNIST za pomocą splotowej sieci neuronowej (CNN) w PyTorch.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torchvision
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

mnist_train = torchvision.datasets.MNIST(root="./data", train=True, download=True)
mnist_test = torchvision.datasets.MNIST(root="./data", train=False, download=True)

X_train_full = mnist_train.data.numpy()
y_train_full = mnist_train.targets.numpy()
X_test = mnist_test.data.numpy()
y_test = mnist_test.targets.numpy()

# Normalizacja pikseli do zakresu [0, 1]
X_train_norm = X_train_full.astype("float32") / 255.0
X_test_norm = X_test.astype("float32") / 255.0

# Podział treningu na train/val (stratyfikowany, 10000 przykładów walidacyjnych)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_norm, y_train_full, test_size=10000, random_state=42, stratify=y_train_full
)

print("Dane załadowane")
print(f"Urządzenie: {device}")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test_norm.shape}")

100.0%
100.0%
100.0%
100.0%

Dane załadowane
Urządzenie: cuda
Train: (50000, 28, 28), Val: (10000, 28, 28), Test: (10000, 28, 28)


In [2]:
# KROK 1: Przygotowanie tensorów i DataLoaderów
# CNN oczekuje danych w kształcie (N, kanały, wysokość, szerokość) = (N, 1, 28, 28)
from torch.utils.data import TensorDataset, DataLoader

print("Kształt obrazów:", X_train.shape)


def to_dataset(X, y):
    X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # dodanie wymiaru kanału
    y_t = torch.tensor(y, dtype=torch.long)
    return TensorDataset(X_t, y_t)


train_ds = to_dataset(X_train, y_train)
val_ds = to_dataset(X_val, y_val)
test_ds = to_dataset(X_test_norm, y_test)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Kształt tensora wejściowego (1 batch):", next(iter(train_loader))[0].shape)

Kształt obrazów: (50000, 28, 28)
Kształt tensora wejściowego (1 batch): torch.Size([32, 1, 28, 28])


In [3]:
# KROK 2: Budowa modelu CNN
# wejście (1, 28, 28) → conv 32 → pool → conv 64 → pool → 128 → wyjście (10)


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),  # (32, 28, 28)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # (32, 14, 14)
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # (64, 14, 14)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # (64, 7, 7)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),  # surowe logity (softmax jest w funkcji straty)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = CNN().to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nLiczba parametrów do nauczenia: {n_params:,}")

CNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

Liczba parametrów do nauczenia: 421,642


In [4]:
# KROK 3: Funkcja straty i optymalizator
# nn.CrossEntropyLoss łączy log-softmax + NLLLoss, więc softmax stosujemy
# wewnątrz straty, a model zwraca surowe logity.
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("Model gotowy do treningu")
print(f"Strata: {criterion.__class__.__name__}")
print(f"Optymalizator: {optimizer.__class__.__name__}")

Model gotowy do treningu
Strata: CrossEntropyLoss
Optymalizator: AdamW


In [5]:
# KROK 4: Trening modelu
EPOCHS = 20


def train_one_epoch():
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += X_batch.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        total_loss += loss.item() * X_batch.size(0)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += X_batch.size(0)
    return total_loss / total, correct / total


history = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = evaluate(val_loader)

    history["loss"].append(train_loss)
    history["accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(
        f"Epoka {epoch:2d}/{EPOCHS}  "
        f"loss: {train_loss:.4f}  acc: {train_acc:.4f}  "
        f"val_loss: {val_loss:.4f}  val_acc: {val_acc:.4f}"
    )

Epoka  1/20  loss: 0.1610  acc: 0.9499  val_loss: 0.0675  val_acc: 0.9800
Epoka  2/20  loss: 0.0468  acc: 0.9853  val_loss: 0.0591  val_acc: 0.9820
Epoka  3/20  loss: 0.0320  acc: 0.9898  val_loss: 0.0420  val_acc: 0.9887
Epoka  4/20  loss: 0.0234  acc: 0.9921  val_loss: 0.0472  val_acc: 0.9869
Epoka  5/20  loss: 0.0188  acc: 0.9936  val_loss: 0.0440  val_acc: 0.9883
Epoka  6/20  loss: 0.0133  acc: 0.9959  val_loss: 0.0412  val_acc: 0.9886
Epoka  7/20  loss: 0.0115  acc: 0.9962  val_loss: 0.0497  val_acc: 0.9866
Epoka  8/20  loss: 0.0094  acc: 0.9968  val_loss: 0.0541  val_acc: 0.9882
Epoka  9/20  loss: 0.0078  acc: 0.9973  val_loss: 0.0543  val_acc: 0.9884
Epoka 10/20  loss: 0.0067  acc: 0.9977  val_loss: 0.0521  val_acc: 0.9887
Epoka 11/20  loss: 0.0062  acc: 0.9980  val_loss: 0.0581  val_acc: 0.9874
Epoka 12/20  loss: 0.0059  acc: 0.9980  val_loss: 0.0546  val_acc: 0.9888
Epoka 13/20  loss: 0.0056  acc: 0.9982  val_loss: 0.0439  val_acc: 0.9901
Epoka 14/20  loss: 0.0041  acc: 0.9986

In [8]:
# KROK 5: Wykresy procesu uczenia (Plotly)
epochs = list(range(1, len(history["loss"]) + 1))

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Dokładność w trakcie treningu",
        "Funkcja straty w trakcie treningu",
    ),
)

# Dokładność
fig.add_trace(
    go.Scatter(
        x=epochs, y=history["accuracy"], mode="lines+markers", name="treningowa"
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=epochs, y=history["val_accuracy"], mode="lines+markers", name="walidacyjna"
    ),
    row=1,
    col=1,
)

# Strata
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=history["loss"],
        mode="lines+markers",
        name="treningowa",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=history["val_loss"],
        mode="lines+markers",
        name="walidacyjna",
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Epoka", row=1, col=1)
fig.update_xaxes(title_text="Epoka", row=1, col=2)
fig.update_yaxes(title_text="Dokładność", row=1, col=1)
fig.update_yaxes(title_text="Strata", row=1, col=2)
fig.update_layout(width=1000, height=450, title_text="Proces uczenia CNN")
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [7]:
# KROK 6: Ocena na zbiorze testowym
test_loss, test_acc = evaluate(test_loader)

print(f"Dokładność na zbiorze testowym: {test_acc:.4f} ({test_acc * 100:.2f}%)")
print(f"Strata na zbiorze testowym:     {test_loss:.4f}")

Dokładność na zbiorze testowym: 0.9910 (99.10%)
Strata na zbiorze testowym:     0.0429


In [9]:
torch.save(model.state_dict(), "model_cnn.pth")
print("Model zapisany do: model_cnn.pth")

# Podgląd zawartości state_dict (nazwa warstwy -> kształt tensora)
for name, tensor in model.state_dict().items():
    print(f"  {name:28s} {tuple(tensor.shape)}")

# Wczytanie modelu z powrotem:
#   model_wczytany = CNN().to(device)
#   model_wczytany.load_state_dict(torch.load("model_cnn.pth", map_location=device))
#   model_wczytany.eval()

Model zapisany do: model_cnn.pth
  features.0.weight            (32, 1, 3, 3)
  features.0.bias              (32,)
  features.3.weight            (64, 32, 3, 3)
  features.3.bias              (64,)
  classifier.1.weight          (128, 3136)
  classifier.1.bias            (128,)
  classifier.3.weight          (10, 128)
  classifier.3.bias            (10,)
